In [1]:
"""
Leakage-Free Preprocessing Pipeline with ColumnTransformer and Cross-Validation
DUT AI Board - Data Science & Machine Learning Course
"""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
RANDOM_STATE = 42

In [2]:
def build_synthetic_churn_dataset(n_samples: int = 1200) -> pd.DataFrame:
    """Create a mixed-type tabular dataset with missing values."""
    rng = np.random.default_rng(RANDOM_STATE)
    tenure_months = rng.integers(1, 60, size=n_samples).astype(float)
    monthly_fee = rng.normal(50, 15, size=n_samples).clip(10, 120)
    support_calls = rng.poisson(lam=1.8, size=n_samples).astype(float)
    contract_type = rng.choice(["Monthly", "Annual", "Two-Year"], size=n_samples, p=[0.5, 0.3, 0.2])

    # Generate binary target from pre-outcome features only
    logits = -0.04 * tenure_months + 0.03 * monthly_fee + 0.45 * support_calls - 1.5
    probabilities = 1.0 / (1.0 + np.exp(-logits))
    churn = rng.binomial(1, probabilities)

    df = pd.DataFrame(
        {
            "tenure_months": tenure_months,
            "monthly_fee": monthly_fee,
            "support_calls": support_calls,
            "contract_type": contract_type,
            "churn": churn,
        }
    )

    # Inject 8% missing values into numerical columns
    missing_mask = rng.random(n_samples) < 0.08
    df.loc[missing_mask, "monthly_fee"] = np.nan
    return df

In [ ]:

def create_leakage_free_pipeline(num_cols: list[str], cat_cols: list[str]) -> Pipeline:
    """Build an end-to-end Scikit-Learn Pipeline that isolates preprocessing."""
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, num_cols),
            ("cat", categorical_pipeline, cat_cols),
        ]
    )
    #preprocessor.fit(X_train)  # Fit the preprocessor on the training data only+ pick model
    model_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )
    return model_pipeline

In [4]:
def main() -> None:
    df = build_synthetic_churn_dataset()
    X = df.drop(columns=["churn"])
    y = df["churn"]

    # Step 1: Split BEFORE any imputation, scaling, or encoding
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    num_cols = ["tenure_months", "monthly_fee", "support_calls"]
    cat_cols = ["contract_type"]
    pipeline = create_leakage_free_pipeline(num_cols, cat_cols)

    # Step 2: Cross-validation fits preprocessing strictly inside each training fold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_auc = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"5-Fold CV ROC-AUC (Train): {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

    # Step 3: Fit once on full training split, then transform and evaluate on sealed Test set
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    print(f"Holdout Test ROC-AUC     : {roc_auc_score(y_test, y_prob):.4f}\n")
    print("Classification Report (Test Set):")
    print(classification_report(y_test, y_pred, digits=4))

if __name__ == "__main__":
    main()

5-Fold CV ROC-AUC (Train): 0.7425 +/- 0.0122
Holdout Test ROC-AUC     : 0.6859

Classification Report (Test Set):
              precision    recall  f1-score   support

           0     0.6890    0.7902    0.7362       143
           1     0.6053    0.4742    0.5318        97

    accuracy                         0.6625       240
   macro avg     0.6471    0.6322    0.6340       240
weighted avg     0.6552    0.6625    0.6536       240

